# 🌐 Indic Multimodal Document Intelligence & Security Guardrails with Gemini 2.0 Flash

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Indic_Multimodal_Document_Intelligence_with_Gemini_2.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-gemini/cookbook/blob/main/examples/Indic_Multimodal_Document_Intelligence_with_Gemini_2.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

<br/><br/>

**Author:** Nandhakumar Murugan ([@nandhakumar-murugan](https://github.com/nandhakumar-murugan)) • Google Student Ambassador  
**Model:** `gemini-2.0-flash`  
**SDK:** `google-genai`  

---

### 📖 Overview
Engineering teams and students globally often encounter complex technical architecture diagrams, flowcharts, and system schemas in English. This recipe shows how to use **Google Gemini 2.0 Flash** to:
1. **Extract and interpret technical diagrams and documents** using high-resolution vision.
2. **Produce structured multilingual explanations** in **Tamil (தமிழ்)**, **Hindi (हिन्दी)**, and **English**.
3. **Enforce Type Safety** with Pydantic structured output (`response_schema`).
4. **Perform automated Cyber Security posture audits** to detect prompt injection risks, sensitive credentials, and architecture vulnerabilities.

## 🛠️ Setup & Installation

In [ ]:
!pip install -q -U google-genai pillow pydantic

### Configure API Key

In [ ]:
import os
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List
from PIL import Image, ImageDraw

# Set your Gemini API key (from Google AI Studio: https://aistudio.google.com)
# In Google Colab, use userdata: from google.colab import userdata; api_key = userdata.get('GEMINI_API_KEY')
api_key = os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
client = genai.Client(api_key=api_key)

## 📐 1. Define Structured Output Schema (Pydantic)

We define a structured schema to guarantee that Gemini returns validated bilingual summaries, concept breakdowns, and a security audit.

In [ ]:
class ConceptDefinition(BaseModel):
    concept: str = Field(description="The technical concept name")
    english_definition: str = Field(description="Clear English explanation")
    tamil_definition: str = Field(description="Precise Tamil (தமிழ்) translation and breakdown")
    hindi_definition: str = Field(description="Precise Hindi (हिन्दी) translation and breakdown")

class DocumentSecurityAssessment(BaseModel):
    contains_sensitive_data: bool = Field(description="Whether the diagram exposes unencrypted keys or PII")
    security_risk_level: str = Field(description="Risk level: Low, Medium, High, or Critical")
    security_observations: List[str] = Field(description="Key security findings or architectural notes")

class IndicDocumentAnalysisResult(BaseModel):
    document_title: str = Field(description="Title of the document or architecture diagram")
    executive_summary_en: str = Field(description="Executive summary in English")
    executive_summary_ta: str = Field(description="Executive summary in Tamil (தமிழ்)")
    executive_summary_hi: str = Field(description="Executive summary in Hindi (हिन्दी)")
    key_concepts: List[ConceptDefinition] = Field(description="Key technical concepts in 3 languages")
    security_assessment: DocumentSecurityAssessment = Field(description="Cyber Security posture assessment")

## 🎨 2. Create Sample Architecture Diagram

In [ ]:
# Generate a sample cloud & security architecture diagram for testing
width, height = 900, 480
img = Image.new("RGB", (width, height), color="#0f172a")
draw = ImageDraw.Draw(img)

# Header
draw.rectangle([(0, 0), (width, 50)], fill="#1e293b")
draw.text((20, 18), "Google Cloud & Zero-Trust Security Architecture Overview", fill="#ffffff")

# Layer 1: Client
draw.rounded_rectangle([(40, 90), (260, 200)], radius=10, fill="#1e3a8a", outline="#3b82f6", width=2)
draw.text((60, 110), "[Client Layer]", fill="#93c5fd")
draw.text((60, 140), "* Web & Flutter App", fill="#ffffff")
draw.text((60, 170), "* OAuth 2.0 / JWT Auth", fill="#cbd5e1")

# Layer 2: Gemini 2.0 Gateway
draw.rounded_rectangle([(340, 90), (580, 200)], radius=10, fill="#064e3b", outline="#10b981", width=2)
draw.text((360, 110), "[AI Gateway Layer]", fill="#6ee7b7")
draw.text((360, 140), "* Google Gemini 2.0 Flash", fill="#ffffff")
draw.text((360, 170), "* Vertex AI Model Armor", fill="#cbd5e1")

# Layer 3: Serverless Backend
draw.rounded_rectangle([(660, 90), (860, 200)], radius=10, fill="#701a75", outline="#d946ef", width=2)
draw.text((680, 110), "[Backend Layer]", fill="#f5d0fe")
draw.text((680, 140), "* Cloud Run Serverless", fill="#ffffff")
draw.text((680, 170), "* Cloud Spanner DB", fill="#cbd5e1")

# Security Policies Box
draw.rounded_rectangle([(40, 240), (860, 430)], radius=10, fill="#1e293b", outline="#f59e0b", width=2)
draw.text((60, 260), "🛡️ Zero-Trust Security Policies (SLSA Level 3 Compliant):", fill="#fbbf24")
draw.text((60, 300), "1. End-to-End mTLS encryption between all microservices.", fill="#e2e8f0")
draw.text((60, 340), "2. Automated OSS-Fuzz static vulnerability scanning on CI/CD.", fill="#e2e8f0")
draw.text((60, 380), "3. Real-time prompt injection detection & PII scrubbing with Model Armor.", fill="#e2e8f0")

img.save("cloud_architecture_sample.png")
display(img)

## ⚡ 3. Execute Multimodal Analysis with Gemini 2.0 Flash

In [ ]:
system_instruction = (
    "You are an expert AI Solution Architect and Cyber Security Engineer. "
    "Your task is to analyze the provided technical architecture diagram. "
    "Extract all architectural components, core concepts, and security policies. "
    "Provide rich, accurate, and fluent explanations in English, Tamil (தமிழ்), and Hindi (हिन्दी). "
    "Perform a thorough Cyber Security posture assessment."
)

prompt = (
    "Analyze this architecture diagram in detail. "
    "Extract the executive summary in English, Tamil, and Hindi. "
    "Define all key technical concepts and evaluate the security architecture."
)

test_image = Image.open("cloud_architecture_sample.png")

# Call Gemini 2.0 Flash
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=[test_image, prompt],
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json",
        response_schema=IndicDocumentAnalysisResult,
        temperature=0.2,
    ),
)

# Parsed Pydantic object
result: IndicDocumentAnalysisResult = response.parsed
print(f"Document Title: {result.document_title}")

## 📊 4. Inspect Results

In [ ]:
print("=" * 60)
print("🇬🇧 ENGLISH SUMMARY:")
print(result.executive_summary_en)
print("\n" + "=" * 60)
print("🇮🇳 TAMIL SUMMARY (தமிழ் விளக்கம்):")
print(result.executive_summary_ta)
print("\n" + "=" * 60)
print("🇮🇳 HINDI SUMMARY (हिन्दी सारांश):")
print(result.executive_summary_hi)
print("\n" + "=" * 60)
print("🛡️ SECURITY ASSESSMENT:")
print(f"Risk Level: {result.security_assessment.security_risk_level}")
for obs in result.security_assessment.security_observations:
    print(f" - {obs}")

## 🎓 Conclusion & Next Steps

By combining **Gemini 2.0 Flash's multimodal vision** with **Pydantic structured output**, developers can bridge language barriers across India while maintaining strict enterprise security standards.

- Explore more at the [Google Gemini Cookbook](https://github.com/google-gemini/cookbook)
- Check out the [google-genai SDK Documentation](https://github.com/googleapis/python-genai)